<a href="https://colab.research.google.com/github/emanaak04-svg/sleep-apnea-detection/blob/main/amna/model_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 01: Install Libraries and Mount Google Drive
We install required libraries and connect Google Drive
to access the merged feature dataset.

In [1]:
!pip install xgboost imbalanced-learn -q

from google.colab import drive
drive.mount('/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported")

Mounted at /drive
All libraries imported


## 02: Load the Merged Feature Dataset
We load the final merged_features.csv.
This dataset has 2754 windows and 23 features combining
ECG HRV, CNN probability, EEG band power, and respiratory
features.

In [2]:
df = pd.read_csv("/drive/MyDrive/shhs-data/merged_features.csv")

print(f"Dataset loaded")
print(f"Shape : {df.shape}")
print(f"Normal: {np.sum(df['label']==0)}")
print(f"Apnea : {np.sum(df['label']==1)}")
print(f"\nFeature columns: {list(df.columns[:-1])}")

Dataset loaded
Shape : (2754, 24)
Normal: 2389
Apnea : 365

Feature columns: ['mean_rr', 'sdnn', 'rmssd', 'mean_hr', 'pnn50', 'lf_hf_ratio', 'cnn_prob', 'delta_power', 'theta_power', 'alpha_power', 'beta_power', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'delta_beta_ratio', 'breathing_rate', 'mean_amplitude', 'std_amplitude', 'mean_peak_dist', 'breath_regularity', 'apnea_index', 'ie_ratio']


## 03: Split Data and Apply SMOTE
We split the data into 80% training and 20% testing using
the exact same random_state as the Random Forest model so
both models are evaluated on the same test set for a fair
comparison. Then we apply SMOTE only on training data.

In [4]:
X = df.drop(columns=['label'])
y = df['label']

# Same split as Random Forest (same random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Before SMOTE — Train: Normal={np.sum(y_train==0)}, Apnea={np.sum(y_train==1)}")

# Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"After SMOTE  — Train: Normal={np.sum(y_train_smote==0)}, Apnea={np.sum(y_train_smote==1)}")
print(f"\nTest set (untouched): Normal={np.sum(y_test==0)}, Apnea={np.sum(y_test==1)}")

Before SMOTE — Train: Normal=1911, Apnea=292
After SMOTE  — Train: Normal=1911, Apnea=1911

Test set (untouched): Normal=478, Apnea=73
Before SMOTE — Train: Normal=1911, Apnea=292
After SMOTE  — Train: Normal=1911, Apnea=1911

Test set (untouched): Normal=478, Apnea=73


## 04: Train XGBoost Classifier
We train an XGBoost classifier on the SMOTE-balanced training
data. XGBoost builds decision trees sequentially, where each
new tree corrects the errors of previous trees. It often
achieves higher accuracy than Random Forest on tabular data.

In [5]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_smote, y_train_smote)

# Predict on test set
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred_xgb)

print(f"XGBoost trained")
print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Normal', 'Apnea']))

XGBoost trained

Test Accuracy: 0.8875 (88.75%)

Classification Report:
              precision    recall  f1-score   support

      Normal       0.94      0.93      0.94       478
       Apnea       0.57      0.59      0.58        73

    accuracy                           0.89       551
   macro avg       0.76      0.76      0.76       551
weighted avg       0.89      0.89      0.89       551



## 05: Save the Trained Model and Predictions
We save the XGBoost model and its predictions so we can use
them for confusion matrix, ROC curve, and the
feature importance plot comparison with Random Forest.

In [6]:
import joblib

# Save model
joblib.dump(xgb_model, "/drive/MyDrive/shhs-data/xgb_model.pkl")

# Save predictions and test data
np.save("/drive/MyDrive/shhs-data/xgb_y_test.npy", y_test.values)
np.save("/drive/MyDrive/shhs-data/xgb_y_pred.npy", y_pred_xgb)
np.save("/drive/MyDrive/shhs-data/xgb_y_pred_proba.npy", y_pred_proba_xgb)

# Save feature importances
feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance_df.to_csv("/drive/MyDrive/shhs-data/xgb_feature_importance.csv", index=False)

print(f"Model and results saved")
print(f"\nTop 10 most important features:")
print(feature_importance_df.head(10).to_string(index=False))

Model and results saved

Top 10 most important features:
          feature  importance
         cnn_prob    0.284764
    std_amplitude    0.095657
 delta_beta_ratio    0.068109
        rel_delta    0.039890
   mean_amplitude    0.037257
   mean_peak_dist    0.036994
breath_regularity    0.036926
      theta_power    0.035493
      apnea_index    0.029656
      delta_power    0.029413
